# JEMM LGU Filipino-English Llama 3.1 LoRA Fine-Tuning

### Install Dependencies

In [1]:
import os, re
if False and "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth  # Do this in local & cloud setups
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
    !pip install --no-deps --upgrade "torchao>=0.16.0"
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2
!pip install langchain_community langchain_huggingface langchain_text_splitters faiss-cpu 

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 27.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 86.7 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.7/119.7 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 199.3/199.3 kB 10.2 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2026.2.0
    Uninstalling fsspec-2026.2.0:
      Successfully uninstalled fsspec-2026.2.0
  Attempting uninstall: dill
    Found existing installation: dill 0.4.1
    Uninstalling dill-0.4.1:
      Successfully uninstalled dill-0.4.1
  Attempting uninstall: datasets
    Found existing installation: datasets 4.8.3
    Uninstalling datasets-4.8.3:
      Successfully uninstalled datasets-4.8.3
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.35.0 requires google-cl

## Data Pre-Processing



In [2]:
import pandas as pd
import re

### Raw Text Q&A Literal Splitting

> Most question rows do not contain sufficient context due to literal splitting.

In [3]:
def open_file_and_split():
  file_path = '/content/FAQs.txt' # Upload file or mount GDrive

  with open(file_path, 'r', encoding='utf-8') as f:
      # Read the whole file and split it by the literal 'Q:' marker
      content = f.read()
      # Splitting by 'Q:' creates a list of blocks
      blocks = content.split('Q:')
      return blocks

In [4]:
def merge_split_faqs():
  blocks = open_file_and_split()
  data = []

  for block in blocks:
      if not block.strip():
          continue  # Skip empty blocks (like the one before the first Q:)

      # Within each block, the first part is the Question,
      # and everything after the first "empty line" is the Answer.
      # We split by double newline (\n\n)
      parts = block.strip().split('\n\n', 1)

      if len(parts) == 2:
          question = parts[0].strip()
          answer = parts[1].strip().replace('\n', ' ')
          data.append({"Question": question, "Answer": answer})
      else:
          # Fallback: if no double newline, use the first single newline
          parts = block.strip().split('\n', 1)
          if len(parts) == 2:
              data.append({"Question": parts[0].strip(), "Answer": parts[1].strip().replace('\n', ' ')})

In [5]:
def save_faq_df():
  df = pd.DataFrame(data)
  output_path = '/content/FAQs.csv'
  df.to_csv(output_path, index=False, encoding='utf-8-sig')

  df.head()

### Import Basic Q&A

In [6]:
jem_qna_df = pd.read_csv("hf://datasets/AjelmarMedina/Philippines_FAQs_Alpaca/Philippines_FAQs_Alpaca.csv")
jem_qna_df = jem_qna_df.rename(columns={
    'Question': 'instruction',
    'Answer': 'output'
})
jem_qna_df.insert(1, 'input', "")

print(jem_qna_df.shape)
jem_qna_df.head()

(340, 3)


,instruction,input,output
0,Ano ang Philippine Local Government Interactiv...,,Ito ay pampublikong datos na may kinalaman sa ...
1,Ano ang layunin ng proyekto?,,Layunin nitong palakasin ang bukas at responsa...
2,Sino ang nasa likod ng proyekto?,,Ang proyekto ay bahagi ng Program on Social an...
3,Sino ang UP CIDS?,,Ang UP CIDS ay itinatag noong 1985 ni dating U...
4,Sino ang PSPC?,,Ang Program on Social and Political Change (PS...


Import pre-made external Alpaca-format dataset from San Nicolas LGU by [clarenceeda](https://hf.co/clarenceeda)

In [7]:
ext_sn_df = pd.read_json("hf://datasets/clarenceeda/sn-lgu-dataset-alpaca/dataset_alpaca.json")
print(ext_sn_df.shape)
ext_sn_df.head()

(557, 3)


,instruction,input,output
0,[Context: SANNICOLAS LGU HELP] What is the pur...,,This service provides officials and employees ...
1,[Context: SANNICOLAS LGU HELP] Who may avail o...,,The service is available to:\n1. All officials...
2,[Context: SANNICOLAS LGU HELP] What are the re...,,### Checklist of Requirements:\n1. **Official ...
3,[Context: SANNICOLAS LGU HELP] What are the cl...,,### Client Steps:\n1. **File the Request:**\n ...
4,[Context: SANNICOLAS LGU HELP] What are the fe...,,### Fees:\n- **Document Fee:** PHP 150.00 (exc...


In [8]:
df = pd.concat([jem_qna_df, ext_sn_df], ignore_index=True)
print(df.shape)
df.head()

(897, 3)


,instruction,input,output
0,Ano ang Philippine Local Government Interactiv...,,Ito ay pampublikong datos na may kinalaman sa ...
1,Ano ang layunin ng proyekto?,,Layunin nitong palakasin ang bukas at responsa...
2,Sino ang nasa likod ng proyekto?,,Ang proyekto ay bahagi ng Program on Social an...
3,Sino ang UP CIDS?,,Ang UP CIDS ay itinatag noong 1985 ni dating U...
4,Sino ang PSPC?,,Ang Program on Social and Political Change (PS...


## LoRA Fine-Tune Training
Via [PLTAT's documentation](https://huggingface.co/PLTAT/Filipino_llama_3.1_FT_8B_GGUF).

In [9]:
from unsloth import FastLanguageModel
import torch
from trl import SFTConfig, SFTTrainer
from datasets import load_dataset, Dataset

max_seq_length = 2048
load_in_4bit = True # QLoRA
lora_rank = 16 # 16GB recommended. Compensate for 15GB of Colab's Free VRAM

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


2026-05-17 12:04:38.218691: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1779019478.419646      57 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1779019478.481756      57 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1779019478.964674      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779019478.964727      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779019478.964731      57 computation_placer.cc:177] computation placer alr

🦥 Unsloth Zoo will now patch everything to make training faster!


### 1. Load the BASE model (Safetensors)

In [15]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "PLTAT/filipino_llama_3.1_FT_lora",
    max_seq_length = max_seq_length,
    dtype = None, # Auto detection
    load_in_4bit = load_in_4bit,
    # device_map = "balanced"
)

==((====))==  Unsloth 2026.5.2: Fast Llama patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


### 2. Add LoRA adapters

In [17]:
model = FastLanguageModel.get_peft_model(
    model,
    r = lora_rank,
    target_modules =["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = lora_rank,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)


Unsloth: Already have LoRA adapters! We shall skip this step.


### 3. Format dataset to Alpaca

In [18]:
alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""

EOS_TOKEN = tokenizer.eos_token
def formatting_prompts_func(examples):
  instructions = examples["instruction"]
  inputs       = examples["input"]
  outputs      = examples["output"]
  texts =[]
  for instruction, input, output in zip(instructions, inputs, outputs):
      text = alpaca_prompt.format(instruction, input, output) + EOS_TOKEN
      texts.append(text)
  return { "text" : texts, }

# Format new Filipino-English dataset
dataset = Dataset.from_pandas(df)
dataset = dataset.map(formatting_prompts_func, batched=True, remove_columns=["instruction", "input", "output"])
dataset = dataset.train_test_split(test_size=0.2, seed=42)


Map:   0%|          | 0/897 [00:00<?, ? examples/s]

### 4. Setup Trainer

In [20]:
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset['train'],
    eval_dataset = dataset['test'],
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    args = SFTConfig(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        # warmup_steps = 5, # 5-10% of total steps
        # max_steps = 60, # Practice Trining
        num_train_epochs = 3, # Training proper
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",

        # Evaluation
        per_device_eval_batch_size = 2,
        eval_accumulation_steps = 4,
        eval_strategy = "steps",
        eval_steps = 15,
    ),
)


Unsloth: Tokenizing ["text"] (num_proc=8):   0%|          | 0/717 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=8):   0%|          | 0/180 [00:00<?, ? examples/s]

### 5. Start Training


In [21]:
is_train_model = True # @param {"type":"boolean"}

if is_train_model:
  trainer_stats = trainer.train()


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 717 | Num Epochs = 3 | Total steps = 270
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 41,943,040 of 8,072,204,288 (0.52% trained)


Step,Training Loss,Validation Loss
15,1.329000,1.309993
30,1.104700,1.122424
45,1.141300,1.051042
60,1.048900,1.000546
75,0.920200,0.960476
90,1.137700,0.927371
105,0.772200,0.913886
120,0.605500,0.890492
135,0.907900,0.871450
150,0.707900,0.857788


In [29]:
# alpaca_prompt = Copied from above
FastLanguageModel.for_inference(model) # Enable native 2x faster inference
inputs = tokenizer(
[
    alpaca_prompt.format(
        "paano kumuha ng solo parent id", # instruction
        "", # input
        "", # output - leave this blank for generation!
    )
], return_tensors = "pt").to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer)
_ = model.generate(**inputs, streamer = text_streamer, max_new_tokens = 512)

<|begin_of_text|>Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
paano kumuha ng solo parent id

### Input:


### Response:
The process for obtaining a Solo Parent ID is as follows: 1. Submit the required documents to the DSWD Field Office. 2. Wait for the verification of your documents and the issuance of the Solo Parent ID. 3. Receive the Solo Parent ID.<|end_of_text|>


In [72]:
is_export_model = False # @param {"type":"boolean","placeholder":"Export trained model to GGUF?"}

if is_export_model:
  model.save_pretrained_gguf("model", tokenizer, quantization_method = "q4_k_m")


## RAG Pipeline

In [84]:
from datasets import load_dataset

ds = load_dataset("AjelmarMedina/JEMM-LGU-RAG", split='train').to_pandas()
ds.head()

,text
0,﻿Philippine Driver's License Renewal - LTOExam...
1,RENEWAL REQUIREMENTS
2,For Non-Professional (NPDL) / Professional (PD...
3,
4,


In [100]:
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Load embedding model (BGE-M3 handles English & Filipino natively)
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
    model_kwargs={'device': 'cpu'} # It's small enough to fit on the T4 now!
)

# Prepare localized knowledge document
text_splitter = RecursiveCharacterTextSplitter(chunk_size=256, chunk_overlap=32)

# Define batch size
BATCH_SIZE = 1024 
vector_db = None

# Iterate through the Hugging Face dataset in batches
num_rows = len(ds)
for i in range(0, num_rows, BATCH_SIZE):
    print(f"Processing rows {i} to {min(i + BATCH_SIZE, num_rows)} out of {num_rows}...")
    
    # Extract just the text column for the current batch
    batch_texts = ds['text'].iloc[i : i + BATCH_SIZE].tolist()    
    docs = text_splitter.create_documents(batch_texts)
    
    if vector_db is None:
        vector_db = FAISS.from_documents(docs, embedding_model)
    else:
        batch_vector_db = FAISS.from_documents(docs, embedding_model)
        vector_db.merge_from(batch_vector_db)

Processing rows 0 to 1024 out of 1952...
Processing rows 1024 to 1952 out of 1952...


In [101]:
def retrieve_context(query: str, k: int = 4):
    retriever = vector_db.as_retriever(search_kwargs={"k": k})
    relevant_docs = retriever.invoke(query)
    context_list = [doc.page_content for doc in relevant_docs]
    formatted_context = "\n\n---\n\n".join(context_list)
    
    return formatted_context

### RAG Generation

In [105]:
user_query = "How to get birth certificate?"

rag_inputs = tokenizer(
[
    alpaca_prompt.format(
        user_query,
        retrieve_context(user_query), # Retrieval-Augmentation
        "", # output - leave this blank for generation!
    )
], return_tensors = "pt").to("cuda")

text_streamer = TextStreamer(tokenizer)
_ = model.generate(**rag_inputs, streamer = text_streamer, max_new_tokens = 512)

<|begin_of_text|>Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
How to get birth certificate?

### Input:
Birth Certificate

---

Birth certificate is not yet registered and the father acknowledges the child at the back of the birth certificate or in a separate public instrument

---

* Civil registry office where the birth certificate is registered

---

* Civil registry office where the birth certificate is registered

### Response:
1. **Birth certificate is not yet registered and the father acknowledges the child at the back of the birth certificate or in a separate public instrument** – The birth certificate will be registered by the civil registrar of the place where the child was born.   **Processing time**: 1 minute to verify the document and 15 minutes to 1 hour to register the document and issue the certificate.   **Fee**: ₱100.00 (birth certificate)